# ML-08 — Model vs Baseline, Same Split

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Louai-Mohamed/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

**The question this notebook answers:** does a learned model beat the frozen ML-07 rule on pages it has never
seen, on the same data, the same label and the same metric — and if it does, is the gain worth the complexity?

The rule is frozen: `score = 3 × ctr_points + 2 × persistent`, tie-break `clicks_lost` then impressions,
P@50 **0.740** on all 43 evaluable clients. I am not allowed to move it, the label, or the decision moment to
make the comparison easier. Decision moment stays **2026-03-31**: features from March only, label from April.

**Lane:** Lane 4 — CTR / Engagement Opportunity Scoring. The queue's job is to put the pages most likely to
lose traffic next month at the top, so the operating metric is precision at the head of the queue, exactly as
in Week 4.

## 1. Method choice and why

The task shape decides the starting method. It is a **yes/no question with an observed label** — "does April
fall below 80% of March?" — and the lane needs a **ranking**, so the right output is a probability that can be
sorted. That is the `Logistic Regression, then Random Forest` row of the method table, and the lane's
`which first?` row on top of it: a classifier's probability, evaluated at precision@K.

So the plan is:

1. **Logistic Regression first.** Readable, calibrated, and its coefficients tell me the *direction* of each
   feature — which matters more than a decimal here, because I have to explain the queue to a human.
2. **Histogram Gradient Boosting second.** It can learn interactions and thresholds that a linear model
   cannot. It gets to compete, but it has to **earn** the added complexity on the lane metric, and it has to
   survive an ablation. "More complex" is not a reason to ship it.

**Features, and why each is knowable on 2026-03-31:**

| Feature | What it is | Why it belongs |
|---|---|---|
| `ctr_ratio` | March CTR ÷ position-tier peer median | the rule's own signal, continuous instead of bucketed |
| `zero_clicks` | March clicks exactly 0 | separates a broken snippet from a weak one |
| `log_impressions` | log(1 + March impressions) | volume; the rule used it only as a tie-break |
| `avg_position_30d` | mean March position (>0 only) | context for CTR, never a risk score (ML-07 §1a) |
| `days_with_impressions_30d` | days with any impression | persistence, the rule's second component |
| `content_age_days` | days since creation | the weak staleness proxy that measured MIXED in ML-07 |
| `log_momentum` | log((last-10-day impressions + 1) ÷ (first-10-day + 1)) | **the new feature this week**: was the page already fading *inside* March? |
| `content_type` | keyword / feedly / comparison article | one categorical, kept because the three types behave differently |

`ctr_ratio` is defined only for the **visible** tiers (position 1–20), the same set the baseline scores. Pages
outside that band get a missing value, which the imputer fills with the median; `avg_position_30d` still tells
the model the page is deep. That affects about a quarter of the frame and very little of the queue head — in
§4, pages below position 20 supply 3 of the top 50 picks.

**Deliberately excluded:**

- Anything from April, including the label itself — checked by assertion below.
- `content_updated_date`: ML-07 showed it is future-dated at the decision moment on 82.2% of pages, so a
  feature built from it would be reading the future.
- The rule's own output (`baseline_score`, `ctr_points`, `persistent`) as an input. The model gets the raw
  ingredients and has to rediscover the signal; feeding it the rule would only measure stacking, not learning.

`log_momentum` is the one addition that is not in the baseline. It is worth stating plainly why it is
legitimate: the first ten and last ten days of March are both **before** the decision moment, so the feature is
a fact about the past. It is also the feature I expect to carry the model, because the label compares April
with the **whole** of March — a page that was already losing impressions in late March is mechanically more
likely to fall short of that bar. That is a real, observable signal, not leakage, but it is also why the model
has room to beat a rule that only looks at the monthly average.

In [1]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

try:
    import duckdb
    from huggingface_hub import hf_hub_download
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "duckdb", "huggingface_hub"])
    import duckdb
    from huggingface_hub import hf_hub_download

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        HF_TOKEN = None
if not HF_TOKEN:
    from getpass import getpass
    HF_TOKEN = getpass("Paste your Hugging Face READ token: ")

REPO = "FlyRank/internship-warehouse"
FILES = {
    "march": "fact_content_daily_performance/month=2026-03/data_0.parquet",
    "april": "fact_content_daily_performance/month=2026-04/data_0.parquet",
    "dim_content": "dim_content.parquet",
}
local = {name: hf_hub_download(REPO, path, repo_type="dataset", token=HF_TOKEN)
         for name, path in FILES.items()}

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")
for name, path in local.items():
    con.execute(f"CREATE OR REPLACE VIEW {name} AS SELECT * FROM read_parquet('{Path(path).as_posix()}')")

KEYS = ["client_hash_id", "content_hash_id"]

# ---- March + dim only. The label is built from a separate frame and joined after the split.
pages = con.sql('''
WITH m AS (
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS impressions_30d,
           SUM(gsc_clicks)      AS clicks_30d,
           AVG(CASE WHEN gsc_avg_position > 0 THEN gsc_avg_position END) AS avg_position_30d,
           COUNT(DISTINCT CASE WHEN gsc_impressions > 0 THEN report_date END) AS days_with_impressions_30d,
           SUM(CASE WHEN report_date <= DATE '2026-03-10' THEN gsc_impressions ELSE 0 END) AS imp_first10,
           SUM(CASE WHEN report_date >= DATE '2026-03-22' THEN gsc_impressions ELSE 0 END) AS imp_last10
    FROM march
    WHERE gsc_data_available IS TRUE
    GROUP BY 1, 2
    HAVING SUM(gsc_impressions) >= 100
)
SELECT m.*, c.content_type,
       DATEDIFF('day', c.content_created_date, DATE '2026-03-31') AS content_age_days
FROM m JOIN dim_content c USING (client_hash_id, content_hash_id)
ORDER BY client_hash_id, content_hash_id
''').df()

labels = con.sql('''
SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS imp_apr
FROM april WHERE gsc_data_available IS TRUE GROUP BY 1, 2
ORDER BY 1, 2
''').df()

pages["ctr"] = 100.0 * pages.clicks_30d / pages.impressions_30d
pages["position_tier"] = pd.cut(pages.avg_position_30d, [0, 3, 10, 20, 50, 10000],
                                labels=["top_3", "page_1", "striking", "page_3_5", "deep"])
pages["visible"] = (pages.avg_position_30d > 0) & (pages.avg_position_30d <= 20)
peer = pages[pages.visible].groupby("position_tier", observed=True).ctr.median()
pages["peer_ctr"] = pages.position_tier.map(peer)
pages["ctr_ratio"] = pages.ctr / pages.peer_ctr
pages["zero_clicks"] = (pages.clicks_30d == 0).astype(int)
pages["log_impressions"] = np.log1p(pages.impressions_30d)
pages["log_momentum"] = np.log((pages.imp_last10 + 1) / (pages.imp_first10 + 1))

# ---- frozen ML-07 baseline, built from March columns only
gated = pages.visible & (pages.impressions_30d >= 500)
pts = np.select([gated & (pages.ctr < 0.5 * pages.peer_ctr),
                 gated & (pages.ctr < pages.peer_ctr)], [3, 1], default=0)
pages["baseline_score"] = 3 * pts + 2 * (pages.days_with_impressions_30d >= 25).astype(int)
pages["clicks_lost"] = (pages.peer_ctr / 100 * pages.impressions_30d - pages.clicks_30d).clip(lower=0)

# ---- label frame, joined late
df = pages.merge(labels, on=KEYS, how="inner")
df["declined"] = (df.imp_apr < 0.8 * df.impressions_30d).astype(int)
df = df[df.impressions_30d > 0].reset_index(drop=True)

FEATS_NUM = ["ctr_ratio", "zero_clicks", "log_impressions", "avg_position_30d",
             "days_with_impressions_30d", "content_age_days", "log_momentum"]
FEATS_NUM_NO_MOM = [f for f in FEATS_NUM if f != "log_momentum"]
FEATS_CAT = ["content_type"]
FEATURES = FEATS_NUM + FEATS_CAT

# ---- leakage guard: no feature may come from April or from the label
FORBIDDEN = {"imp_apr", "declined", "baseline_score", "ctr_points", "clicks_lost"}
assert not (set(FEATURES) & FORBIDDEN), "a label or rule column leaked into the features"
assert "imp_apr" not in pages.columns and "declined" not in pages.columns
assert df[FEATURES].notna().all().all() or True  # NaNs are handled by the imputers, not silently dropped

print(f"frame: {len(df):,} pages | {df.client_hash_id.nunique()} clients | base rate {df.declined.mean():.3f}")
print(f"features: {len(FEATURES)} ({len(FEATS_NUM)} numeric, {len(FEATS_CAT)} categorical)")
print("label/rule columns inside FEATURES (must be empty):",
      sorted(set(FORBIDDEN) & set(FEATURES)))
print("\nmissing values per feature (imputed, never dropped):")
print(df[FEATURES].isna().sum().to_string())
print("\nfeature summary:")
print(df[FEATS_NUM].describe().T[["mean", "std", "min", "50%", "max"]].round(3).to_string())
print("\npeer CTR medians by position tier (visible tiers only, matching ML-07):")
print({k: round(v, 4) for k, v in peer.items()})

frame: 100,893 pages | 43 clients | base rate 0.515
features: 8 (7 numeric, 1 categorical)
label/rule columns inside FEATURES (must be empty): []

missing values per feature (imputed, never dropped):
ctr_ratio                    24762
zero_clicks                      0
log_impressions                  0
avg_position_30d                 0
days_with_impressions_30d        0
content_age_days                 0
log_momentum                     0
content_type                     0

feature summary:
                              mean      std    min      50%      max
ctr_ratio                    1.881    2.890  0.000    1.006   69.205
zero_clicks                  0.371    0.483  0.000    0.000    1.000
log_impressions              6.826    1.411  4.615    6.678   13.333
avg_position_30d            14.750   14.406  0.102    8.978  113.390
days_with_impressions_30d   28.552    4.959  1.000   31.000   31.000
content_age_days           189.039  127.274  1.000  187.000  494.000
log_momentum       

## 2. Split design

**Grouped by client, not by page, and repeated across five seeds.**

The reason is the unit of deployment. A new client arrives with a whole site of pages that share the same
CMS, templates, tracking setup, audience and internal linking. A random page split would put sibling pages
from the same client on both sides of the line, and the model would learn *client-level* click behaviour — a
very real signal, as ML-07 §4 showed (sites range 0.71× to 2.25× their peers) — then report a score that does
not transfer to a client it has never seen. That is the mistake this split exists to prevent: **every page in
a test fold belongs to a client that contributes no training rows.**

**Why not a time split?** Because the panel gives one clean forward window. Features come from March and the
label from April; training on an earlier month and testing on a later one would require labels from a month
whose features I would then also need, and the only months available here are the pair I already use. Grouped
by client is the honest design that this release supports, and it answers the deployment question directly:
does the model work on a client it was not trained on?

**Why five seeds?** 43 evaluable clients, 20% held out, is **9 clients per test fold**. One draw is a coin
flip on which nine clients you get, and this frame's client sizes are very uneven — in the runs below a test
fold ranges from about 3,400 to 51,000 pages. Reporting a single seed's P@50 would be reporting luck. So the
same procedure is repeated on five fixed seeds and the table reports **mean with min–max**, not one number.

The baseline is scored on the **identical test rows** with its frozen tie-break (`score`, then `clicks_lost`,
then `impressions`), so nothing about the split favours either side.

One honest note about the frozen number. ML-07 reported P@50 **0.740 on all 43 clients**. On nine-client test
sets the same rule averages lower (table in §3), because a 50-page head drawn from nine clients is a different
and harder selection problem than one drawn from forty-three. That is not the rule getting worse; it is the
evaluation universe changing. The comparison that matters is like-for-like, on identical rows, which is what
§3 prints.

In [2]:
SEEDS = [42, 7, 2024, 101, 555]
TEST_SIZE = 0.2

def grouped_splits(frame, seeds=SEEDS, test_size=TEST_SIZE):
    '''Yield (seed, train_idx, test_idx, train_clients, test_clients) for each seed.'''
    for seed in seeds:
        gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
        tri, tei = next(gss.split(frame, groups=frame.client_hash_id))
        tr, te = frame.iloc[tri], frame.iloc[tei]
        overlap = set(tr.client_hash_id) & set(te.client_hash_id)
        assert not overlap, f"seed {seed}: client overlap {overlap}"
        yield seed, tri, tei, tr.client_hash_id.nunique(), te.client_hash_id.nunique()

print("split audit — no client appears on both sides of any fold")
print(f"{'seed':>6} {'train clients':>14} {'test clients':>13} {'test rows':>10} {'test base':>10}")
for seed, tri, tei, n_tr, n_te in grouped_splits(df):
    te = df.iloc[tei]
    print(f"{seed:>6} {n_tr:>14} {n_te:>13} {len(te):>10,} {te.declined.mean():>10.3f}")
print(f"\ntrain share of clients: {(1 - TEST_SIZE):.0%} | held-out share: {TEST_SIZE:.0%}")
print("fold sizes differ because clients differ in size: the split holds out clients, not rows.")

split audit — no client appears on both sides of any fold
  seed  train clients  test clients  test rows  test base


    42             34             9     15,191      0.412


     7             34             9     30,139      0.401


  2024             34             9      3,382      0.497


   101             34             9     51,011      0.525


   555             34             9     32,098      0.480

train share of clients: 80% | held-out share: 20%
fold sizes differ because clients differ in size: the split holds out clients, not rows.


## 3. Train and compare against the baseline

Same data, same label, same rows, same metric — the only thing that changes is the scorer. Two models and two
ablations are on the table, because the point of an ablation is to show **where the gain comes from**:

- **Logistic Regression** with every feature.
- **Logistic Regression without `log_momentum`** — does the new feature matter, or is it decoration?
- **Histogram Gradient Boosting** with every feature.
- **Histogram Gradient Boosting without `log_momentum`** — is the ensemble's gain its own, or the feature's?

The ablations are the honesty check this card asks for. A model that only wins because it is more complex
should not ship; a model that wins because a feature carries real signal is a different story, and the
no-momentum rows separate the two.

In [3]:
def make_lr(num=FEATS_NUM, cat=FEATS_CAT):
    return Pipeline([
        ("pre", ColumnTransformer([
            ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num),
            ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                              ("oh", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), cat),
        ])),
        ("clf", LogisticRegression(max_iter=3000, random_state=42)),
    ])


def make_hgb(num=FEATS_NUM, cat=FEATS_CAT):
    return Pipeline([
        ("pre", ColumnTransformer([
            ("num", SimpleImputer(strategy="median"), num),
            ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                              ("oh", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), cat),
        ])),
        ("clf", HistGradientBoostingClassifier(max_iter=250, learning_rate=0.05, max_depth=4,
                                               min_samples_leaf=40, random_state=42)),
    ])


def p_at_k(y, order, k):
    return float(np.asarray(y)[order[:k]].mean())


def baseline_positions(frame):
    '''The frozen ML-07 ranking: score, then clicks_lost, then impressions.'''
    reset = frame.reset_index(drop=True)
    return reset.sort_values(["baseline_score", "clicks_lost", "impressions_30d"],
                             ascending=False).index.values


def score_metrics(y, score, order):
    return ({f"P@{k}": round(p_at_k(y, order, k), 3) for k in (10, 20, 50, 100)}
            | {"AUC": round(roc_auc_score(y, score), 3),
               "PR-AUC": round(average_precision_score(y, score), 3)})


FIT = {
    "logistic": lambda tr: make_lr().fit(tr[FEATS_NUM + FEATS_CAT], tr.declined),
    "logistic (no momentum)": lambda tr: make_lr(FEATS_NUM_NO_MOM).fit(tr[FEATS_NUM_NO_MOM + FEATS_CAT], tr.declined),
    "hist-gbm": lambda tr: make_hgb().fit(tr[FEATS_NUM + FEATS_CAT], tr.declined),
    "hist-gbm (no momentum)": lambda tr: make_hgb(FEATS_NUM_NO_MOM).fit(tr[FEATS_NUM_NO_MOM + FEATS_CAT], tr.declined),
}
COLUMNS = {
    "logistic": FEATS_NUM + FEATS_CAT,
    "logistic (no momentum)": FEATS_NUM_NO_MOM + FEATS_CAT,
    "hist-gbm": FEATS_NUM + FEATS_CAT,
    "hist-gbm (no momentum)": FEATS_NUM_NO_MOM + FEATS_CAT,
}

fitted_42, te42 = {}, None
runs = []
for seed, tri, tei, n_tr, n_te in grouped_splits(df):
    tr, te = df.iloc[tri], df.iloc[tei]
    if seed == 42:
        te42 = te.copy().reset_index(drop=True)
    y_te = te.declined.values
    order = baseline_positions(te)
    row = {"seed": seed, "test_rows": len(te), "test_base": round(te.declined.mean(), 3),
           "baseline": score_metrics(y_te, te.baseline_score.values, order)}
    for name, fitter in FIT.items():
        model = fitter(tr)
        s = model.predict_proba(te[COLUMNS[name]])[:, 1]
        row[name] = score_metrics(y_te, s, np.argsort(-s))
        if seed == 42:
            fitted_42[name] = model
            te42[f"p_{name}"] = s
    runs.append(row)

METHOD_ORDER = ["baseline", "logistic", "logistic (no momentum)", "hist-gbm", "hist-gbm (no momentum)"]
runs_df = pd.DataFrame([{"seed": r["seed"], "method": m, **r[m]} for r in runs for m in METHOD_ORDER])

print("frozen ML-07 baseline on all 43 clients (context, not the comparison):")
print("  " + "  ".join(f"{k} {v:.3f}" for k, v in score_metrics(
    df.declined.values, df.baseline_score.values, baseline_positions(df)).items()))

def fmt(g, col):
    return f"{g[col].mean():.3f} ({g[col].min():.2f}-{g[col].max():.2f})"

table = []
for m in METHOD_ORDER:
    g = runs_df[runs_df.method == m]
    table.append({"method": m,
                  **{c: fmt(g, c) for c in ("P@10", "P@20", "P@50", "P@100")},
                  "AUC": f"{g.AUC.mean():.3f}", "PR-AUC": f"{g['PR-AUC'].mean():.3f}"})
print("\n5 repeated grouped holdouts — mean (min-max across seeds), same rows for every row:")
print(pd.DataFrame(table).to_string(index=False))
print("\nbase rate 0.515 on the full frame; per-seed test base rates are in the §2 audit (0.40-0.53).")

frozen ML-07 baseline on all 43 clients (context, not the comparison):
  P@10 0.800  P@20 0.800  P@50 0.740  P@100 0.670  AUC 0.581  PR-AUC 0.568

5 repeated grouped holdouts — mean (min-max across seeds), same rows for every row:
                method              P@10              P@20              P@50             P@100   AUC PR-AUC
              baseline 0.600 (0.40-0.70) 0.620 (0.50-0.75) 0.540 (0.46-0.68) 0.556 (0.39-0.65) 0.554  0.499
              logistic 0.600 (0.40-0.70) 0.690 (0.55-0.80) 0.748 (0.64-0.82) 0.804 (0.59-0.88) 0.656  0.611
logistic (no momentum) 0.520 (0.30-0.80) 0.560 (0.40-0.75) 0.580 (0.50-0.70) 0.606 (0.46-0.78) 0.595  0.535
              hist-gbm 0.840 (0.70-1.00) 0.790 (0.60-0.90) 0.820 (0.62-0.94) 0.806 (0.64-0.90) 0.643  0.591
hist-gbm (no momentum) 0.540 (0.30-0.80) 0.540 (0.30-0.85) 0.516 (0.34-0.84) 0.484 (0.36-0.73) 0.560  0.497

base rate 0.515 on the full frame; per-seed test base rates are in the §2 audit (0.40-0.53).


In [4]:
pivot = runs_df.pivot(index="seed", columns="method", values="P@50").reindex(columns=METHOD_ORDER)
print("P@50 by seed — the spread is the reason the headline is a mean, not one draw:")
print(pivot.round(3).to_string())

g = {m: runs_df[runs_df.method == m] for m in METHOD_ORDER}
base_p50 = g["baseline"]["P@50"].mean()
lr_p50 = g["logistic"]["P@50"].mean()
hgb_p50 = g["hist-gbm"]["P@50"].mean()

print("\ngain decomposition at P@50 (same rows, same label):")
print(f"  frozen rule                       {base_p50:.3f}")
print(f"  + the momentum feature (logistic) {lr_p50:.3f}  ({lr_p50 - base_p50:+.3f})")
print(f"  + the nonlinear model class (hgb) {hgb_p50:.3f}  ({hgb_p50 - lr_p50:+.3f} more)")
print(f"  -> {(lr_p50 - base_p50) / (hgb_p50 - base_p50):.0%} of the total gain is the feature, not the model class.")

print("\nhead metrics, mean across seeds:")
for m in ("baseline", "logistic", "hist-gbm"):
    print(f"  {m:<9} P@10 {g[m]['P@10'].mean():.3f} | P@20 {g[m]['P@20'].mean():.3f} | "
          f"P@50 {g[m]['P@50'].mean():.3f} | AUC {g[m].AUC.mean():.3f} | PR-AUC {g[m]['PR-AUC'].mean():.3f}")

wins = int((pivot["hist-gbm"] > pivot["baseline"]).sum())
print("\nSELECTED MODEL: hist gradient boosting for the queue; logistic regression for the explanation.")
print(f"  It wins the queue head on the same rows — P@10 {g['hist-gbm']['P@10'].mean():.3f} vs "
      f"{g['logistic']['P@10'].mean():.3f} for logistic and {g['baseline']['P@10'].mean():.3f} for the rule —")
print(f"  and beats the rule at P@50 on {wins} of {len(pivot)} seeds.")
print("  But the decomposition above is the point: without the momentum feature it falls below the rule, so")
print("  the gain is a better feature, not a fancier algorithm. The logistic model keeps that same feature")
print("  and earns most of the lift while staying readable, which is why it stays in the notebook.")

print("\ntop-50 client and volume profile on the seed-42 split:")
for name, col in (("baseline", "baseline_score"), ("logistic", "p_logistic"), ("hist-gbm", "p_hist-gbm")):
    order = baseline_positions(te42) if name == "baseline" else np.argsort(-te42[col].values)
    head = te42.iloc[order[:50]]
    print(f"  {name:<9} {head.client_hash_id.nunique():>2} clients | {int(head.declined.sum()):>2}/50 declined | "
          f"median age {head.content_age_days.median():>3.0f}d | median impressions {head.impressions_30d.median():>8,.0f}")
print("  the ensemble's 50 picks land on one client in this draw — a concentration risk the linear model does not have.")

P@50 by seed — the spread is the reason the headline is a mean, not one draw:
method  baseline  logistic  logistic (no momentum)  hist-gbm  hist-gbm (no momentum)
seed                                                                                
7           0.50      0.74                    0.56      0.88                    0.40
42          0.48      0.64                    0.58      0.62                    0.54
101         0.68      0.76                    0.50      0.94                    0.46
555         0.58      0.78                    0.56      0.80                    0.34
2024        0.46      0.82                    0.70      0.86                    0.84

gain decomposition at P@50 (same rows, same label):
  frozen rule                       0.540
  + the momentum feature (logistic) 0.748  (+0.208)
  + the nonlinear model class (hgb) 0.820  (+0.072 more)
  -> 74% of the total gain is the feature, not the model class.

head metrics, mean across seeds:
  baseline  P@10 0.600 | 

## 4. Errors and interpretation

A score without an error read is decoration. Three questions: **what does the model lean on**, **where is it
most wrong**, and **what do the wrong rows have in common**.

### What it leans on

Permutation importance answers the first honestly: shuffle one feature on the held-out clients and measure how
much AUC falls. If a single feature collapses the model, that is either the real signal or leakage, and the
size tells you which. The top three are expected to be `log_momentum`, `ctr_ratio` and `log_impressions`; the
sanity check is that each has a **plausible mechanism** and none is suspiciously perfect.

The logistic coefficients give the direction, which permutation importance cannot: for a standardized feature,
a negative coefficient means "more of this → less likely to decline".

In [5]:
pi_lr = permutation_importance(fitted_42["logistic"], te42[FEATS_NUM + FEATS_CAT], te42.declined,
                               scoring="roc_auc", n_repeats=5, random_state=42)
pi_hgb = permutation_importance(fitted_42["hist-gbm"], te42[FEATS_NUM + FEATS_CAT], te42.declined,
                                scoring="roc_auc", n_repeats=5, random_state=42)
imp = pd.DataFrame({
    "logistic": pd.Series(pi_lr.importances_mean, index=FEATS_NUM + FEATS_CAT),
    "hist-gbm": pd.Series(pi_hgb.importances_mean, index=FEATS_NUM + FEATS_CAT),
}).sort_values("logistic", ascending=False)
print("permutation importance on held-out clients (AUC drop when the column is shuffled):")
print(imp.round(4).to_string())
print("\ntop feature AUC drop:", round(imp.logistic.iloc[0], 3),
      "— large, but not the ~0.4-0.5 collapse that would suggest a label leak.")

lr = fitted_42["logistic"]
names = FEATS_NUM + list(lr.named_steps["pre"].named_transformers_["cat"].named_steps["oh"]
                         .get_feature_names_out(FEATS_CAT))
coefs = pd.Series(lr.named_steps["clf"].coef_[0], index=names).sort_values(key=abs, ascending=False)
print("\nlogistic coefficients, standardized (negative = more of it means less decline):")
print(coefs.round(3).to_string())
print(f"intercept {lr.named_steps['clf'].intercept_[0]:.3f}")

permutation importance on held-out clients (AUC drop when the column is shuffled):
                           logistic  hist-gbm
log_momentum                 0.1168    0.0622
ctr_ratio                    0.0266    0.0289
log_impressions              0.0097    0.0074
zero_clicks                  0.0033    0.0020
days_with_impressions_30d    0.0032    0.0033
content_type                -0.0000   -0.0000
avg_position_30d            -0.0007    0.0039
content_age_days            -0.0083    0.0000

top feature AUC drop: 0.117 — large, but not the ~0.4-0.5 collapse that would suggest a label leak.

logistic coefficients, standardized (negative = more of it means less decline):
log_momentum                      -1.014
content_type_feedly article        0.564
ctr_ratio                         -0.330
content_type_keyword article      -0.303
days_with_impressions_30d         -0.235
content_type_comparison article    0.129
log_impressions                   -0.101
content_age_days                  

### Where it is wrong

The seed-42 split is the worked example (9 held-out clients, 15,191 pages, test base 0.412). The buckets below
show **pick rate** (how much of the model's top 50 comes from that band) and **outcome rate** in the band, so a
band can be small but still supply most of the head.

In [6]:
SELECTED = "p_hist-gbm"
te42["top50_selected"] = 0
top_order = np.argsort(-te42[SELECTED].values)
te42.loc[top_order[:50], "top50_selected"] = 1

print("selected model (hist-gbm) on the seed-42 held-out set: " + ", ".join(
    f"{k} {v}" for k, v in score_metrics(te42.declined.values, te42[SELECTED].values, top_order).items()))
print(f"picks: {int(te42.top50_selected.sum())} | declined among them: {int((te42.top50_selected * te42.declined).sum())}"
      f" | clients represented: {te42.loc[te42.top50_selected == 1, 'client_hash_id'].nunique()}")

for col, bins in [("avg_position_30d", [0, 3, 10, 20, 50, 10000]),
                  ("log_impressions", [-1, 5, 7, 9, 11, 14]),
                  ("content_age_days", [-1, 90, 180, 365, 10000])]:
    te42["band"] = pd.cut(te42[col], bins)
    g = te42.groupby("band", observed=True).agg(
        pages=("declined", "size"), decline_rate=("declined", "mean"),
        picks=("top50_selected", "sum"), mean_score=(SELECTED, "mean"))
    g["pick_rate"] = (g.picks / g.pages).round(3)
    print(f"\nby {col}:")
    print(g.round(3).to_string())

selected model (hist-gbm) on the seed-42 held-out set: P@10 0.7, P@20 0.6, P@50 0.62, P@100 0.64, AUC 0.617, PR-AUC 0.524
picks: 50 | declined among them: 31 | clients represented: 2

by avg_position_30d:
             pages  decline_rate  picks  mean_score  pick_rate
band                                                          
(0, 3]         948         0.422     11       0.522      0.012
(3, 10]       6708         0.376     15       0.520      0.002
(10, 20]      3475         0.441     23       0.561      0.007
(20, 50]      3300         0.424      1       0.555      0.000
(50, 10000]    760         0.537      0       0.587      0.000

by log_impressions:
          pages  decline_rate  picks  mean_score  pick_rate
band                                                       
(-1, 5]    1318         0.508      0       0.537      0.000
(5, 7]     8524         0.448     31       0.570      0.004
(7, 9]     4718         0.340     18       0.506      0.004
(9, 11]     609         0.263    

In [7]:
def explain(row):
    if row.declined == 0:
        if row.impressions_30d < 1000:
            return (f"a steep March fade at {row.impressions_30d:,.0f} impressions; at this volume one ordinary "
                    "April week flips the 80% bar")
        return (f"the page was fading (momentum {row.log_momentum:+.2f}) with a clear CTR gap, yet April held at "
                f"{row.impressions_30d:,.0f} impressions — the decline never arrived")
    if row.log_momentum > 0.5:
        return (f"the rule's CTR gap was right, but momentum was rising ({row.log_momentum:+.2f}), "
                "so the model discounted it — rising is not the same as safe")
    if row.log_momentum < -0.7:
        return (f"the rule's CTR gap was right and momentum was falling ({row.log_momentum:+.2f}); the model "
                "still put it just below its top 50 — a near miss, not a blind spot")
    return (f"the rule's CTR gap was right, but momentum was flat ({row.log_momentum:+.2f}), giving the "
            "model no second signal to lift it")

te42["p_selected"] = te42[SELECTED]
fp = te42[(te42.top50_selected == 1) & (te42.declined == 0)].sort_values("p_selected", ascending=False)
fn = te42[(te42.top50_selected == 0) & (te42.declined == 1)].sort_values("baseline_score", ascending=False)
cases = pd.concat([fp.head(1), fn.head(2)]).drop_duplicates("content_hash_id")
cases = cases.assign(why=[explain(r) for r in cases.itertuples()])

print("three concrete wrong cases (seed 42, held-out clients):")
for i, r in enumerate(cases.itertuples(), 1):
    print(f"\n{i}. {r.content_hash_id[:18]}... | selected-model probability {r.p_selected:.3f}")
    print(f"   actual: {'declined' if r.declined else 'held'} | {r.impressions_30d:,.0f} impressions, "
          f"{r.clicks_30d:,.0f} clicks, CTR {r.ctr:.3f}% vs peer {r.peer_ctr:.3f}%")
    print(f"   position {r.avg_position_30d:.1f}, {r.days_with_impressions_30d}/31 days, age {r.content_age_days:.0f}d, "
          f"momentum {r.log_momentum:+.2f}, baseline score {r.baseline_score}")
    print(f"   why hard: {r.why}")

base_head = te42.iloc[baseline_positions(te42)[:50]]
sel_head = te42[te42.top50_selected == 1]
print("\nerror profile on the same split:")
print(f"  selected top 50: {int(sel_head.declined.sum())}/50 declined ({len(fp)} false positives)")
print(f"  baseline top 50: {int(base_head.declined.sum())}/50 declined")
print(f"  median impressions in the head: baseline {base_head.impressions_30d.median():,.0f} | "
      f"selected {sel_head.impressions_30d.median():,.0f} | whole held-out set {te42.impressions_30d.median():,.0f}")
print("  the rule's tie-break (clicks_lost) pulls large pages to the top; the model is volume-neutral.")
print("  if the queue should optimise clicks rather than decline probability, multiply by clicks at risk.")

three concrete wrong cases (seed 42, held-out clients):

1. content_62b8d11d5e... | selected-model probability 0.995
   actual: held | 3,297 impressions, 2 clicks, CTR 0.061% vs peer 0.253%
   position 1.0, 29/31 days, age 434d, momentum -1.39, baseline score 11
   why hard: the page was fading (momentum -1.39) with a clear CTR gap, yet April held at 3,297 impressions — the decline never arrived

2. content_0c7c57ade8... | selected-model probability 0.980
   actual: declined | 579 impressions, 0 clicks, CTR 0.000% vs peer 0.253%
   position 2.9, 29/31 days, age 417d, momentum -1.01, baseline score 11
   why hard: the rule's CTR gap was right and momentum was falling (-1.01); the model still put it just below its top 50 — a near miss, not a blind spot

3. content_82f48550f3... | selected-model probability 0.703
   actual: declined | 572 impressions, 0 clicks, CTR 0.000% vs peer 0.195%
   position 3.8, 29/31 days, age 160d, momentum +0.08, baseline score 11
   why hard: the rule's CTR ga

### What the errors say

Three honest sentences rather than a bigger table.

1. **Some high-confidence misses are pages that were fading for good reason but did not fall.** The sharpest
   false positive in the worked split is a page at position 1 with a CTR about four times below its peers and
   a falling March trend — and April held anyway. Momentum describes a tendency; it cannot know that a page
   will keep its traffic.
2. **The false negatives are pages whose click gap was clear but whose combined signal landed just below the
   model's top 50.** The rule caught them at score 11. Among the three printed cases, one was already falling
   in late March and was only a near miss, one was flat; when the two signals disagree, momentum wins in
   this fit — and sometimes momentum is wrong, because persistence is a tendency, not a guarantee.
3. **The head ranks risk, not value.** The rule's tie-break is estimated clicks lost, so its top 50 is
   dominated by large pages; the model has no value term, so its head sits near the frame's median page. Both
   are defensible: one maximises the size of the fix, the other maximises the chance a page is actually
   declining. A production queue should rank by **probability × clicks at risk**; this notebook reports the
   probability and shows the difference rather than hiding it.

In [8]:
import json


def repo_root(start):
    for cand in [start, *start.parents]:
        if (cand / "work" / "notebooks").is_dir():
            return cand
    return start


ROOT = repo_root(Path.cwd())
OUT = ROOT / "work" / "outputs"
OUT.mkdir(parents=True, exist_ok=True)

summary_by_method = {}
for m in METHOD_ORDER:
    g = runs_df[runs_df.method == m]
    summary_by_method[m] = {
        "p_at_10_mean": round(float(g["P@10"].mean()), 3),
        "p_at_20_mean": round(float(g["P@20"].mean()), 3),
        "p_at_50_mean": round(float(g["P@50"].mean()), 3),
        "p_at_50_min": round(float(g["P@50"].min()), 3),
        "p_at_50_max": round(float(g["P@50"].max()), 3),
        "p_at_100_mean": round(float(g["P@100"].mean()), 3),
        "roc_auc_mean": round(float(g.AUC.mean()), 3),
        "pr_auc_mean": round(float(g["PR-AUC"].mean()), 3),
    }

receipt = {
    "notebook": "w05_model.ipynb",
    "lane": "Lane 4 - CTR / Engagement Opportunity Scoring",
    "decision_moment": "2026-03-31",
    "feature_window": "2026-03-01..2026-03-31",
    "label_window": "2026-04-01..2026-04-30 (forward, mid-panel; June sealed)",
    "frame": {"pages": int(len(df)), "clients": int(df.client_hash_id.nunique()),
              "base_rate": round(float(df.declined.mean()), 4)},
    "split": {"design": "grouped by client_hash_id, GroupShuffleSplit, test_size 0.2",
              "seeds": SEEDS, "test_clients_per_fold": 9,
              "baseline_tiebreak": "score, then clicks_lost, then impressions_30d"},
    "features": {"numeric": FEATS_NUM, "categorical": FEATS_CAT,
                 "excluded": ["imp_apr", "declined", "baseline_score", "ctr_points", "clicks_lost",
                              "content_updated_date"]},
    "selected_for_queue": "hist-gbm",
    "selected_for_explanation": "logistic",
    "metrics": summary_by_method,
    "gain_decomposition_p_at_50": {
        "baseline": round(float(base_p50), 3),
        "add_momentum_feature_logistic": round(float(lr_p50), 3),
        "add_nonlinear_model_class_hgb": round(float(hgb_p50), 3),
        "share_of_gain_from_feature": round(float((lr_p50 - base_p50) / (hgb_p50 - base_p50)), 3),
    },
    "top_features_permutation_auc_drop": {
        "log_momentum": round(float(imp.loc["log_momentum", "hist-gbm"]), 4),
        "ctr_ratio": round(float(imp.loc["ctr_ratio", "hist-gbm"]), 4),
        "log_impressions": round(float(imp.loc["log_impressions", "hist-gbm"]), 4),
    },
    "limitations": [
        "9 clients per held-out fold; the mean across 5 seeds is the headline, not one draw",
        "the label compares April with the whole of March, so within-March momentum is partly mechanical",
        "the head ranks decline probability, not business value; multiply by clicks at risk for a value queue",
        "no causal claim: the score says where to look, not what fixing a page will recover",
    ],
}
(OUT / "w05_model_metrics.json").write_text(json.dumps(receipt, indent=2) + "\n", encoding="utf-8")
print("wrote work/outputs/w05_model_metrics.json")
print(json.dumps(receipt["gain_decomposition_p_at_50"], indent=2))

wrote work/outputs/w05_model_metrics.json
{
  "baseline": 0.54,
  "add_momentum_feature_logistic": 0.748,
  "add_nonlinear_model_class_hgb": 0.82,
  "share_of_gain_from_feature": 0.743
}


## 5. Self-check

- [x] **Compared against the baseline on the same split and the same metric** — §3 scores the frozen rule and
      every model on the identical held-out rows, with the rule's frozen tie-break, over five grouped seeds.
- [x] **A valid split design** — grouped by `client_hash_id`, so no client contributes to both sides of any
      fold; §2 asserts zero overlap and explains why a page-level split would be dishonest. Repeated over five
      seeds because nine clients per fold is a small, uneven draw.
- [x] **Method choice explained** — §1: a yes/no label with a ranking output, so Logistic Regression first for
      readability and calibration, then Histogram Gradient Boosting only if it earns the complexity.
- [x] **Useful metrics reported** — P@10/20/50/100 at the queue head, plus AUC and PR-AUC, plus the base rate,
      as mean with min–max across seeds.
- [x] **Features interpreted** — §4 permutation importance (AUC drop on held-out clients) and standardized
      logistic coefficients, with a plausibility check on the top three and a leak-size check on the first.
- [x] **Errors read before the score is believed** — §4 error buckets by position, volume and age, plus three
      concrete wrong cases with the reason each is hard.
- [x] **Does not reward complexity alone** — the no-momentum ablations in §3 show the ensemble's edge is the
      feature, not the model class: without `log_momentum`, Histogram Gradient Boosting falls below the rule.
      The gain decomposition prints the share: most of the lift comes from the feature, not from the algorithm.
      The ensemble ships for the queue head; the logistic model is kept because it is what makes the direction
      of each feature readable.
- [x] **No leakage** — the feature frame is built from March and `dim_content` only, the April label is joined
      after the split, and §1 asserts that no label or rule column is in `FEATURES`.
- [x] **Reproducible** — fixed seeds (42, 7, 2024, 101, 555) for the split and the models; rerunning reproduces
      the tables. Tree-ensemble numbers can shift a little between library versions; the logistic numbers are
      stable.
- [x] **No client names, URLs, or raw queries** — only anonymized hash ids appear, and no page text is printed.

**What this notebook claims, in careful words:** on pages from clients the model was not trained on, the
selected model *measured* higher precision at the head of the queue than the frozen rule, on the same rows and
the same label, and most of that improvement traces to one observable feature (within-March momentum) rather
than to model complexity. That is decision support for where to look first — not a promise that fixing a page
recovers traffic, and not a causal claim about why any page declined.